In [1]:
!pip install gradio transformers tensorflow --quiet

In [2]:
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
model = "bigcode/starcoder2-7b"
tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModelForCausalLM.from_pretrained(
  model,
  torch_dtype=torch.bfloat16,
  device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.51G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.89G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [7]:
bug_model_path = "/content/bug_detection_model.h5"
tokenizer_path = "/content/tokenizer.pkl"
max_len = 300
bug_model = load_model(bug_model_path)

with open(tokenizer_path, "rb") as f:
    code_tokenizer = pickle.load(f)

def detect_bug(code_snippet):
  seq = code_tokenizer.texts_to_sequences([code_snippet])
  seq = pad_sequences(seq, maxlen=max_len)
  pred = bug_model.predict(seq)[0][0]
  if pred > 0.5:
    return f"!Bug Detected (score={pred:.3f})"
  else:
    return f"No Bug Detected (score={pred:.3f})"

In [8]:
def generate_docstring(code_snippet):
  prompt = f"""
### Task:
Write a clean python docstring for the fuction below.
### Code:
{code_snippet}
### Docstring:
\"\"\"
"""
  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  output = model.generate(
    **inputs,
    max_length=256,
    temperature=0.2,
    num_beams=4,
    pad_token_id=tokenizer.eos_token_id
  )

  decoded = tokenizer.decode(output[0], skip_special_tokens=True)
  if '"""' in decoded:
    return decoded.split('"""')[1].strip()

  return decoded

In [9]:
def generate_suggestion(context_code):
    prompt = f"""
### Task:
Provide the next lines of code based on the context.
### Context:
{context_code}
### Suggested code:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_length=200,
        temperature=0.3,
        num_beams=4,
        pad_token_id=tokenizer.eos_token_id
    )
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    if "Suggested code:" in decoded:
        return decoded.split("Suggested code:")[1].strip()
    return decoded


In [10]:
doc_interface = gr.Interface(
  fn=generate_docstring,
  inputs=gr.Textbox(lines=12, label="paste python Code"),
  outputs=gr.Textbox(lines=10, label="genrated Docstring"),
  title="AI docstring Genrator (StarCoder2-7B)"
)

suggest_interface = gr.Interface(
  fn=generate_suggestion,
  inputs=gr.Textbox(lines=12, label="Paste Code Context"),
  outputs=gr.Textbox(lines=10, label="Code Suggestion"),
  title="Context Awre Code Suggestion (StarCoder2-7B)"
)

bug_interface = gr.Interface(
    fn=detect_bug,
    inputs=gr.Textbox(lines=12, label="paste code"),
    outputs="text",
    title="Bug Detction (LSTM+CNN)"
)


# Combine all features into tabs
demo = gr.TabbedInterface(
    [doc_interface, suggest_interface, bug_interface],
    ["Docstring Genrator", "Context Awre Suggession", "Bug Detctor"]
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://626dd74b7bb4115847.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://626dd74b7bb4115847.gradio.live
